# RegRAG on Google Colab

A retrieval-augmented assistant for Indian state renewable-energy regulations: clause-level
chunks, hybrid BM25 + dense retrieval with a state filter, cross-encoder reranking, amendment
checks, and cited answers from Groq `gpt-oss-120b` (embeddings run locally in Colab via Ollama).

This notebook **sets everything up inside Colab** (about 5 minutes), then walks through the
pipeline step by step.

**Before you run**
1. **Runtime → Change runtime type → T4 GPU.** It also works on CPU, but building the index is slower.
2. Have **`regrag_colab.zip`** ready to upload (built with `python scripts/make_colab_bundle.py`),
   **or** set `REPO_URL` below if the project is on GitHub.
3. Have your **Groq API key** ready; you'll be asked for it.
4. Run the cells in order (**Runtime → Run all** works too; it will stop at the upload and key prompts).

## 0.1 Check the runtime

In [ ]:
import shutil, subprocess, sys
print(sys.version)
gpu = shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none - building the index will be slower (Runtime -> Change runtime type -> T4 GPU)")

## 0.2 Get the project code

Either upload `regrag_colab.zip` when prompted, or set `REPO_URL` to a GitHub repository.

In [ ]:
import os, zipfile
from pathlib import Path

REPO_URL = ""                      # e.g. "https://github.com/<you>/regrag.git" - leave empty to upload the zip
PROJECT = Path("/content/regrag")

if not (PROJECT / "regrag").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()                      # choose regrag_colab.zip
        zip_name = next(n for n in uploaded if n.endswith(".zip"))
        with zipfile.ZipFile(zip_name) as z:
            z.extractall(PROJECT)
os.chdir(PROJECT)
print("project at", Path.cwd(), "->", sorted(p.name for p in PROJECT.iterdir())[:12])

## 0.3 Install Python packages (~1–2 min)

In [ ]:
!pip install -q -r requirements-colab.txt

## 0.4 Groq API key

Answers are written by Groq `gpt-oss-120b`. The key stays in this Colab session only.

In [ ]:
import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

## 0.5 Install and start Ollama, pull the embedding model (~2 min)

Ollama serves `nomic-embed-text` (embeddings). On a GPU runtime it uses CUDA automatically.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /content/ollama.log")

In [ ]:
!ollama pull nomic-embed-text > /dev/null 2>&1 && echo "nomic-embed-text ready"

## 0.6 Download the regulation PDFs and build the index

The PDFs come from the source URLs in `data/manifest.json`. On a GPU, building the index takes
about a minute.

In [ ]:
!python scripts/download_docs.py

In [ ]:
from regrag import index, ingest

ingest.run()
index.run()

Setup is done. The rest of this notebook is the **same walkthrough** as the local version.

---

In [ ]:
import json, os, sys, time, warnings, urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "regrag").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from regrag import config

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# Is Ollama running with the models we need?
try:
    tags = json.load(urllib.request.urlopen(f"{config.OLLAMA_BASE_URL}/api/tags", timeout=5))
    have = {m["name"].split(":")[0] for m in tags["models"]}
    print("Ollama is up. Models:", sorted(have))
    needed = {config.EMBED_MODEL} | ({config.LLM_MODEL} if config.LLM_PROVIDER == "ollama" else set())
    missing = needed - have
    print("Missing models:", missing or "none")
except OSError:
    print("Ollama is NOT reachable - start it with `ollama serve` before running the retrieval cells.")

print(f"Project root: {ROOT}")
print(f"LLM: {config.ANSWER_MODEL} ({config.LLM_PROVIDER}) | embeddings: {config.EMBED_MODEL} | reranker: {config.RERANK_MODEL}")
if config.LLM_PROVIDER == "groq":
    print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))

## 1. Documents and metadata

Each PDF is described in `data/manifest.json`. That metadata (state, issuing authority, document
type, date, draft/final status, and which earlier document it amends) travels with every chunk,
and is what later makes state filtering and amendment checks possible.

In [ ]:
from regrag.ingest import load_manifest

docs = pd.DataFrame(load_manifest())
docs["pdf_present"] = docs["file"].map(lambda f: (config.RAW_DIR / f).exists())
docs[["state", "authority", "doc_type", "date", "status", "amends", "pdf_present"]]

Missing PDFs? Download them with `python scripts/download_docs.py`.
One KERC order is a scanned image with no text layer, so the manifest marks it `skip`.

## 2. Clause-level chunking

Regulations are numbered (`6`, `6.2`, `7.3`), so we split on **clause headings** instead of fixed
windows. The splitter only accepts a heading if the numbering moves forward in small steps, which
filters out table rows and quantities like `1 MW`. In amending documents, quoted clauses stay inside
the paragraph that quotes them.

Here is the RERC 2024 order, which raised Rajasthan's net-metering limit:

In [ ]:
from regrag.ingest import amended_refs, extract_lines, split_clauses

lines = extract_lines(config.RAW_DIR / "RJ_RERC_SuoMotu_Order_2024.pdf")
blocks = split_clauses(lines, strict_subclauses=True)
pd.DataFrame([{"clause": b["clause"] or "(preamble)", "page": b["page"],
               "starts_with": " ".join(b["lines"])[:85]} for b in blocks])

For amending documents we also record **which clauses they change**: a clause reference only counts
if an amending verb ("substituted", "increase", ...) appears near it.

In [ ]:
for text in ["Regulation 7 of the Principal Regulations shall be substituted as under",
             "decides to increase the capacity limit ... as specified in regulation 7",
             "Capacity of Solar Rooftop to be installed (Subject to Regulation 6.2)"]:
    print(f"{amended_refs(text)!s:<8} <- {text}")

Now chunk every document. This takes a few seconds and writes `data/processed/chunks.jsonl`.

In [ ]:
from regrag import ingest

chunks = ingest.run()
chunk_df = pd.DataFrame([{"doc_id": c.doc_id, "state": c.state, "clause": c.clause,
                          "chars": len(c.text), "refs": ",".join(c.refs)} for c in chunks])
chunk_df.groupby("state").agg(chunks=("doc_id", "size"), median_chars=("chars", "median"))

## 3. Indexing

Two indexes over the same chunks:
- **Dense**: `nomic-embed-text` (768-dimensional) vectors in ChromaDB, cosine similarity.
- **Keyword**: BM25 over the same text. It's good at exact terms like `500 kW` or `Regulation 7.3`.

Each chunk is indexed with a **contextual header** (state, authority, title, clause), so both indexes
"know" which state a clause belongs to.

In [ ]:
from regrag.index import BM25Index, build_vectorstore, load_vectorstore

REBUILD_INDEX = False   # True re-embeds all chunks (~15 min on CPU)

if REBUILD_INDEX or not config.CHROMA_DIR.exists():
    store = build_vectorstore(chunks)
else:
    store = load_vectorstore()
print("vectors in ChromaDB:", store._collection.count())

print("\nExample contextual text that gets embedded:\n")
print(chunks[0].embed_text[:400])

## 4. Query understanding

We detect the target state(s) from names, regulators, discoms and cities. Comparison questions are
split into one sub-query per state, and questions about states we have no documents for are
declined up front.

In [ ]:
from regrag.query import decompose, detect_states, out_of_scope_message

examples = ["What does BESCOM pay for rooftop solar exports?",
            "Compare the transformer limits in Gujarat and Maharashtra",
            "What is the net metering limit in Kerala?",
            "How does net metering work?"]
rows = []
for q in examples:
    rows.append({"question": q, "states": detect_states(q),
                 "sub_queries": [s.text for s in decompose(q)],
                 "out_of_scope": out_of_scope_message(q) is not None})
pd.DataFrame(rows)

## 5. Hybrid retrieval: BM25 + dense, fused with RRF

Both retrievers run **inside the state filter**, so a Rajasthan question can never retrieve a
Gujarat clause. Their rankings are merged with **reciprocal-rank fusion**:

$$\text{RRF}(d) = \sum_{r \in \{\text{bm25},\,\text{dense}\}} \frac{1}{60 + \text{rank}_r(d)}$$

A chunk ranked well by *both* retrievers rises to the top.

In [ ]:
from regrag.pipeline import components, state_scope

retriever, checker = components()     # loads the chunks, BM25, ChromaDB (reranker loads lazily)

QUESTION = "What is the maximum net metering capacity in Rajasthan?"
sub = decompose(QUESTION)[0]
scope = state_scope(sub, QUESTION)
print("state filter:", scope)

bm25 = [c.chunk_id for c, _ in retriever.bm25.search(QUESTION, scope, 5)]
dense = [c.chunk_id for c in retriever._dense(QUESTION, scope, 5)]
fused = retriever.candidates(QUESTION, scope)
pd.DataFrame({"BM25 top 5": bm25, "Dense top 5": dense,
              "Fused (RRF) top 5": [h.chunk.chunk_id for h in fused[:5]]})

## 6. Cross-encoder reranking

The fused pool (up to 30 chunks) is re-scored by `ms-marco-MiniLM-L-6-v2`, a cross-encoder that
reads the question and chunk **together**. It's slower than the retrievers but more accurate. The
top 5 go to the model. If even the best score is very low, the system abstains without calling
the LLM.

The first run downloads the reranker model (~90 MB).

In [ ]:
t = time.time()
hits = retriever.retrieve(QUESTION, scope)
print(f"reranked in {time.time() - t:.1f}s  (abstain threshold: {config.ABSTAIN_SCORE})")
pd.DataFrame([{"rerank_score": round(h.rerank, 2), "rrf": round(h.rrf, 4), "doc": h.chunk.doc_id,
               "clause": h.chunk.clause, "date": h.chunk.date, "found_by": sorted(h.sources),
               "text": h.chunk.text[:80].replace("\n", " ")} for h in hits])

## 7. Amendment check and temporal ranking

Regulations change. For every retrieved clause we look for **later documents in the same state that
amend that clause**. The older clause is flagged `POSSIBLY SUPERSEDED` (not hidden), any missing
amendment is pulled into the context, and superseded clauses are ranked below current ones.

Gujarat is a good example. The 2016 regulation capped rooftop capacity at 50% of sanctioned load;
the 2017 amendment exempted residential consumers.

In [ ]:
GJ_Q = "Is a residential consumer in Gujarat limited to 50% of sanctioned load for rooftop solar?"
gj_sub = decompose(GJ_Q)[0]
raw_hits = retriever.retrieve(gj_sub.text, state_scope(gj_sub, GJ_Q))
checked = checker.check(gj_sub.text, raw_hits)

pd.DataFrame([{"order": i + 1, "doc": h.chunk.doc_id, "clause": h.chunk.clause, "date": h.chunk.date,
               "rerank": round(h.rerank, 2),
               "flags": " | ".join(f.split(" - ")[0] for f in h.flags) or "-"} for i, h in enumerate(checked)])

The 2016 clause 6.2 is still shown, for traceability, but flagged and ranked last, below the 2017
consolidated version that replaced it.

## 8. Cited answer generation

The model sees the sources split into **CURRENT** and **OLDER OR DRAFT** sections. It must cite
`[S#]` on every sentence, copy numbers exactly, and reply `INSUFFICIENT_CONTEXT` if the sources
don't answer the question. This is the exact prompt context for the Gujarat question:

In [ ]:
from regrag.generate import SYSTEM_PROMPT, format_sources, generate, llm

print(SYSTEM_PROMPT)
print("\n" + "=" * 100 + "\n")
print(format_sources(checked)[:1500], "\n...")

### ⏱ Generate the answer

In [ ]:
t = time.time()
ans = generate(GJ_Q, checked, llm())
print(f"({time.time() - t:.0f}s) abstained={ans.abstained}\n")
print(ans.text)
print()
for c in ans.citations:
    print(f"  [{c['id']}] {c['title']} | clause {c['clause']} | {c['date']} | p.{c['page']}")

## 9. The full pipeline

`pipeline.answer()` chains steps 4–8. Comparison questions run once per state and the answers are
merged with state-tagged citations. Out-of-scope questions return immediately.

In [ ]:
from regrag.pipeline import answer

t = time.time()
kerala = answer("What is the net metering limit for rooftop solar in Kerala?")
print(f"Kerala ({time.time() - t:.1f}s): {kerala.answer}")

### ⏱ A two-state comparison (one generation per state)

In [ ]:
t = time.time()
res = answer("Compare the distribution transformer capacity limits for rooftop solar in Gujarat and Maharashtra.")
print(f"({time.time() - t:.0f}s)\n")
display(Markdown(res.answer))
for c in res.citations:
    print(f"  [{c['id']}] {c['state']} | {c['title'][:70]} | clause {c['clause']} | {c['date']}")

## 10. LangGraph agent

The agent wraps the same steps in a graph with a **self-check**. After generating, an LLM judge
verifies the answer is grounded in the sources. If it isn't, the query is rewritten and retrieval
runs once more. The retry is **rolled back** if it finds worse sources, so a bad rewrite can't
replace a good answer.

In [ ]:
from regrag.graph import build_graph, run as run_agent

print(build_graph().get_graph().draw_mermaid())

### ⏱ Run the agent

The agent makes several LLM calls: generate, verify, and possibly rewrite and regenerate.
Set `RUN_AGENT = False` to skip this cell.

In [ ]:
RUN_AGENT = True
AGENT_QUESTION = "What is the current net metering capacity limit in Rajasthan?"

if RUN_AGENT:
    t = time.time()
    agent_res, trace = run_agent(AGENT_QUESTION)
    print(f"({time.time() - t:.0f}s)\n\nTrace:")
    for step in trace:
        print("  ·", step)
    print()
    display(Markdown(agent_res.answer))
else:
    print("Skipped (RUN_AGENT = False).")

## 11. Evaluation

The evaluation used 34 hand-written questions (`eval/testset.json`), with every reference answer
checked against the PDFs. It was run with `eval/run_eval.py`; the results are loaded from
`eval/results/` below.

In [ ]:
res_dir = ROOT / "eval" / "results"
retrieval = json.loads((res_dir / "retrieval.json").read_text(encoding="utf-8"))["summary"]
ragas = json.loads((res_dir / "ragas_pipeline.json").read_text(encoding="utf-8"))

sys.path.insert(0, str(ROOT / "eval"))
from run_eval import answer_metrics
answers = answer_metrics(False)

summary = {**{f"retrieval: {k}": v for k, v in retrieval.items()},
           **{f"answers: {k}": v for k, v in answers.items() if k != "rows"},
           **{f"ragas: {k}": v for k, v in ragas.items() if k not in ("n_scored", "judge")}}
pd.Series(summary, name="value").to_frame()

In [ ]:
per_q = pd.read_csv(res_dir / "ragas_pipeline.csv")
per_q["group"] = per_q["id"].str[:2]
per_q.groupby("group")[["faithfulness", "llm_context_precision_with_reference", "answer_relevancy"]].mean().round(2)

To re-run the evaluation (slow on CPU):

```bash
python eval/run_eval.py retrieval     # ~2 min, no LLM
python eval/run_eval.py answers       # ~1 h, resumable
python eval/run_eval.py ragas         # needs GROQ_API_KEY in .env
python eval/run_eval.py report
```

### Figures

`scripts/make_figures.py` renders the presentation figures from these same result files.

In [ ]:
for name in ["08_models_and_workflow", "02_retrieval_vs_generation", "03_faithfulness_by_state"]:
    path = ROOT / "docs" / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path), width=900))
    else:
        print("missing:", path.name, "- run `python scripts/make_figures.py`")

## Summary

- **Retrieval is near-perfect.** The right clause reached the model for every test question, with
  0% cross-state contamination, thanks to clause-level chunks, the state filter, hybrid search and
  reranking.
- **Amendments are handled explicitly.** Superseded clauses are flagged and ranked below the
  provisions that replace them.
- **The answer model matters most.** With the local 3B `llama3.2`, RAGAS faithfulness was 0.64.
  Answers now come from Groq `gpt-oss-120b` by default (`LLM_PROVIDER` in `.env`), which reads the
  clauses much more reliably.